# Analyze document data with Pandas

## What you will learn in this course 🧐🧐

MongoDB is a great source for semi-structured data; Pandas is great for analysis.
In this lecture, you’ll connect them end-to-end using the **actual shape** of Atlas’s `sample_analytics`:

By the end, you will be able to:

* Connect with PyMongo
* Paginate & project to avoid pulling the world
* Turn Mongo results into a Pandas DataFrame
* Flatten nested arrays with `pd.json_normalize()`
* Compute KPIs with `groupby`/`agg`
* write results back to MongoDB
* Handle gotchas: `ObjectId`, dates, large result sets, high-precision numeric strings

## Analytics workflow

The classic analytics workflow is:

![](https://full-stack-assets.s3.eu-west-3.amazonaws.com/Classic_analytics_workflow.png)

Let's review it step by step

## Connect to MongoDB (PyMongo)

In [ ]:
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
import pandas as pd

USERNAME = "xxxxxx_db_user" # Replace with your username 
PASSWORD = "xxxxxx_db_password" # Replace with your password
CLUSTER_NAME= "Cluster0" # Replace with your cluster name
MONGODB_URI=f"mongodb+srv://{USERNAME}:{PASSWORD}@{CLUSTER_NAME.lower()}.1tsvfmh.mongodb.net/?retryWrites=true&w=majority&appName={CLUSTER_NAME.lower()}"

client = MongoClient(MONGODB_URI, server_api=ServerApi('1'))

try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

db = client.sample_analytics
print(db.list_collection_names())

Pinged your deployment. You successfully connected to MongoDB!
['accounts', 'customers', 'transactions']


## Query, Projection & Pagination (REAL `customers` schema)

When querying with Pandas, one thing that you want to avoid is to pull the whole database. Indeed Pandas loads data in-memory (meaning on your local machine), so you can't hold too much data. Therefore, what you can do is to both:

* Select fields with projections 
* Create pagination

In [15]:
page_size = 100
page = 0  # 0-based

query = {} # no filter, get all documents
projection = {
    "_id": 0,
    "username": 1,
    "name": 1,
    "email": 1,
    "address": 1,
    "accounts": 1,   # array of account_id (ints)
}

cursor = (
    db.customers.find(query, projection)
    .sort("name", 1)
    .skip(page * page_size) # Skips the first `page * page_size` results
    .limit(page_size)
)

customers = list(cursor)
customers_df = pd.DataFrame(customers)
customers_df.head()

,username,name,address,email,accounts
0,lars.archive,NaN,NaN,NaN,NaN
1,lars.archive,NaN,NaN,NaN,NaN
2,lars.archive,NaN,NaN,NaN,NaN
3,lars.archive,NaN,NaN,NaN,NaN
4,david77,Aaron Perez,55375 Malone Trail Suite 506\nSouth Miguelland...,alexaortega@hotmail.com,"[744220, 126092, 187107, 437371, 413293]"


## Flatten Arrays & Nested Structures

### Flattening VS Exploding

Flattening takes nested structures (like embedded documents or arrays) and turns them into a single, wider record—one row keeps its identity, but nested fields become new columns. 

Exploding takes an array field and produces multiple rows—each item in the array becomes its own row, typically duplicating the parent record’s other fields.

Let's review an example to illustrate both case

### Flatten `transactions` (array inside each document)

Each document in `transactions` holds **one account’s entire transaction array**. Let's flatten it so each **transaction becomes a row**.

In [16]:
from pprint import pprint

# Pull account_id + transactions array
records = list(db.transactions.find({}, {"_id": 0, "account_id": 1, "transactions": 1}))

print("=== Raw documents as stored in MongoDB ===")
pprint(records[0]) # print the first document as an example

# Flatten transactions so each row is a single transaction
transactions_df = pd.json_normalize(
    records,
    record_path="transactions",
    meta=["account_id"]
)

# Cast types once we have a tabular view
transactions_df["amount"] = pd.to_numeric(transactions_df["amount"], errors="coerce")
transactions_df["date"] = pd.to_datetime(transactions_df["date"], errors="coerce", utc=True)
transactions_df["price"] = pd.to_numeric(transactions_df["price"], errors="coerce")
transactions_df["total"] = pd.to_numeric(transactions_df["total"], errors="coerce")

print("=== Flattened transactions (first 5 rows) ===")
transactions_df.head()

=== Raw documents as stored in MongoDB ===
{'account_id': 443178,
 'transactions': [{'amount': 7514,
                   'date': datetime.datetime(2003, 9, 9, 0, 0),
                   'price': '19.1072802650074180519368383102118968963623046875',
                   'symbol': 'adbe',
                   'total': '143572.1039112657392422534031',
                   'transaction_code': 'buy'},
                  {'amount': 9240,
                   'date': datetime.datetime(2016, 6, 14, 0, 0),
                   'price': '24.1525632387771480580340721644461154937744140625',
                   'symbol': 'team',
                   'total': '223169.6843263008480562348268',
                   'transaction_code': 'buy'},
                  {'amount': 2824,
                   'date': datetime.datetime(2002, 12, 4, 0, 0),
                   'price': '21.046193953245431629284212249331176280975341796875',
                   'symbol': 'msft',
                   'total': '59434.45172396509892109861539',
  

,date,amount,transaction_code,symbol,price,total,account_id
0,2003-09-09 00:00:00+00:00,7514,buy,adbe,19.107280,143572.103911,443178
1,2016-06-14 00:00:00+00:00,9240,buy,team,24.152563,223169.684326,443178
2,2002-12-04 00:00:00+00:00,2824,buy,msft,21.046194,59434.451724,443178
3,2014-07-14 00:00:00+00:00,7418,sell,sap,76.385145,566625.008617,443178
4,2011-10-28 00:00:00+00:00,5638,buy,adbe,28.365658,159925.577598,443178


### Link `customers` ↔ `accounts` (explode & merge)

`customers.accounts` is an **array of account IDs**.
Explode it to one row per (customer, account_id), then merge with the `accounts` collection.

In [17]:
# Customers subset
cust = pd.DataFrame(list(db.customers.find(
    {}, {"_id": 0, "username": 1, "name": 1, "address": 1, "accounts": 1}
)))

# One row per account_id
cust_exploded = cust.explode("accounts").rename(columns={"accounts": "account_id"})
cust_exploded["account_id"] = pd.to_numeric(cust_exploded["account_id"], errors="coerce")

# Accounts
acct = pd.DataFrame(list(db.accounts.find(
    {}, {"_id": 0, "account_id": 1, "limit": 1, "products": 1}
)))
acct["account_id"] = pd.to_numeric(acct["account_id"], errors="coerce")

# Join
cust_accounts = cust_exploded.merge(acct, on="account_id", how="left")
cust_accounts.head()

,username,name,address,account_id,limit,products
0,matthewray,John Parks,"38456 Rachael Causeway Apt. 735\nEvanfort, AR ...",702610.0,10000.0,"[Commodity, CurrencyService, InvestmentStock]"
1,matthewray,John Parks,"38456 Rachael Causeway Apt. 735\nEvanfort, AR ...",240640.0,10000.0,"[Brokerage, Derivatives, InvestmentFund, Inves..."
2,thomasdavid,Ashley Lopez,"18637 Jessica Ridge Apt. 157\nGrossberg, ME 84127",662207.0,9000.0,"[Commodity, InvestmentFund, Brokerage, Derivat..."
3,thomasdavid,Ashley Lopez,"18637 Jessica Ridge Apt. 157\nGrossberg, ME 84127",816481.0,10000.0,"[Derivatives, InvestmentStock]"
4,taylorbullock,Shirley Rodriguez,"7637 Johnson Circles\nNew Laurahaven, KY 21914",784245.0,10000.0,[InvestmentStock]


## KPIs (built on the real schema)

Let's review a few example KPIs to practice 

### KPI 1 — **Top products by number of accounts**


In [18]:
prod = cust_accounts.explode("products")
kpi_products = (
    prod.groupby("products")["account_id"]
    .nunique()
    .reset_index(name="num_accounts")
    .sort_values("num_accounts", ascending=False)
)
kpi_products.head()

,products,num_accounts
5,InvestmentStock,1745
2,CurrencyService,741
0,Brokerage,740
4,InvestmentFund,728
1,Commodity,719


### KPI 2 — **Average account limit by product**

In [19]:
avg_limit_by_product = (
    prod.groupby("products")["limit"]
    .mean()
    .reset_index(name="avg_limit")
    .sort_values("avg_limit", ascending=False)
)
avg_limit_by_product.head()

,products,avg_limit
1,Commodity,9963.988920
0,Brokerage,9960.969044
5,InvestmentStock,9955.949657
4,InvestmentFund,9951.923077
3,Derivatives,9951.841360


### KPI 3 — **Total traded amount per symbol**

In [20]:
kpi_symbol_amount = (
    transactions_df.groupby("symbol")["amount"]
    .sum()
    .reset_index()
    .sort_values("amount", ascending=False)
)
kpi_symbol_amount.head()

,symbol,amount
1,adbe,27463715
7,ebay,27232371
5,crm,27099929
9,goog,27029894
14,nvda,26108705


### KPI 4 — **Transaction activity by product** (join tx with acct)

In [21]:
tx_acct = transactions_df.merge(acct, on="account_id", how="left")
tx_prod = tx_acct.explode("products")

kpi_amount_by_product = (
    tx_prod.groupby("products")["amount"]
    .sum()
    .reset_index(name="total_amount_traded")
    .sort_values("total_amount_traded", ascending=False)
)
kpi_amount_by_product.head()

,products,total_amount_traded
5,InvestmentStock,440199918
0,Brokerage,190603512
1,Commodity,182899950
4,InvestmentFund,182817264
2,CurrencyService,181021915


## Write Results Back to MongoDB - If necessary

Store analytics tables for dashboards / downstream tools:

In [22]:
db.analytics_products_summary.insert_many(kpi_products.to_dict("records"))
db.analytics_avg_limit_by_product.insert_many(avg_limit_by_product.to_dict("records"))
db.analytics_symbol_amount.insert_many(kpi_symbol_amount.to_dict("records"))

InsertManyResult([ObjectId('69b3f32523fb93d3456ca324'), ObjectId('69b3f32523fb93d3456ca325'), ObjectId('69b3f32523fb93d3456ca326'), ObjectId('69b3f32523fb93d3456ca327'), ObjectId('69b3f32523fb93d3456ca328'), ObjectId('69b3f32523fb93d3456ca329'), ObjectId('69b3f32523fb93d3456ca32a'), ObjectId('69b3f32523fb93d3456ca32b'), ObjectId('69b3f32523fb93d3456ca32c'), ObjectId('69b3f32523fb93d3456ca32d'), ObjectId('69b3f32523fb93d3456ca32e'), ObjectId('69b3f32523fb93d3456ca32f'), ObjectId('69b3f32523fb93d3456ca330'), ObjectId('69b3f32523fb93d3456ca331'), ObjectId('69b3f32523fb93d3456ca332'), ObjectId('69b3f32523fb93d3456ca333'), ObjectId('69b3f32523fb93d3456ca334'), ObjectId('69b3f32523fb93d3456ca335')], acknowledged=True)

## Gotchas (as they apply to `sample_analytics`) ⚠️

* **`ObjectId`**: if you include `_id`, convert to string for Pandas displays: `str(doc["_id"])`.
* **Dates**: `transactions.date` parses fine; keep `utc=True` for consistent resampling/windowing.
* **Large result sets**: prefer **filtering & projection**, paginate, or aggregate in Mongo first (`$match/$unwind/$group`).
* **High-precision numeric strings**: `price`/`total` are strings with many decimals. For exact math use `decimal.Decimal` or cast in Mongo with `$toDecimal` in an aggregation pipeline.
* **Explode multiplication**: exploding `accounts` or `products` multiplies rows. Do it intentionally, then aggregate.


## Resources 📚📚

* [PyMongo](https://pymongo.readthedocs.io/en/stable/)
* [pandas.json_normalize](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html)
* [Aggregation Pipeline](https://www.mongodb.com/docs/manual/core/aggregation-pipeline/)
* [MongoDB Python Quickstart](https://www.mongodb.com/docs/drivers/pymongo/)
* [Atlas Sample Data](https://www.mongodb.com/docs/atlas/sample-data/)
